In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("/content/campaign_experiment_data.xlsx - experiment_data.csv")  # adjust path to your repo's data/ folder
print(df.shape)
df.head()

(1408, 16)


,user_id,signup_date,experiment_group,region,device_type,traffic_source,plan_type,visited_landing_page,started_trial,completed_onboarding,converted_to_paid,revenue_30d,support_tickets_30d,refund_requested,days_to_convert,engagement_score
0,USR-100001,2025-03-16,Control,North,Desktop,Email,Basic,0,0,0,0,0.0,0,0,NaN,45.7
1,USR-100002,2025-03-08,Control,West,Mobile,Organic,Free,1,0,0,0,0.0,0,0,NaN,48.6
2,USR-100003,2025-04-24,Control,East,Mobile,Paid Search,Premium,1,0,0,0,0.0,1,0,NaN,35.4
3,USR-100004,2025-05-04,Control,South,Mobile,Email,Basic,1,0,0,0,0.0,0,0,NaN,64.6
4,USR-100005,2025-02-12,Control,North,Tablet,Organic,Free,0,0,0,0,0.0,0,0,NaN,66.6


In [2]:
print("Missing values:\n", df.isnull().sum())
print("\nGroup sizes:\n", df['experiment_group'].value_counts())
# days_to_convert is missing for ~95% of rows because it's only defined for
# users who actually converted - this is expected, not a data quality issue.

Missing values:
 user_id                    0
signup_date                0
experiment_group           0
region                     0
device_type               18
traffic_source            24
plan_type                  0
visited_landing_page       0
started_trial              0
completed_onboarding       0
converted_to_paid          0
revenue_30d                0
support_tickets_30d        0
refund_requested           0
days_to_convert         1336
engagement_score          14
dtype: int64

Group sizes:
 experiment_group
Treatment    715
Control      693
Name: count, dtype: int64


In [3]:
def prop_test(df, col, group_col='experiment_group'):
    c = df[df[group_col] == 'Control']
    t = df[df[group_col] == 'Treatment']
    x1, n1 = c[col].sum(), len(c)
    x2, n2 = t[col].sum(), len(t)
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p2 - p1) / se if se > 0 else np.nan
    pval = 2 * (1 - stats.norm.cdf(abs(z))) if se > 0 else np.nan
    lift = (p2 - p1) / p1 * 100 if p1 > 0 else np.inf
    return pd.Series({'control_rate': p1, 'treatment_rate': p2,
                       'relative_lift_%': lift, 'z_stat': z, 'p_value': pval})

funnel_cols = ['visited_landing_page', 'started_trial',
               'completed_onboarding', 'converted_to_paid']
funnel_results = pd.DataFrame({col: prop_test(df, col) for col in funnel_cols}).T
print(funnel_results)
# converted_to_paid: control ~3.2% -> treatment ~7.0%, p < 0.01 -> statistically significant

                      control_rate  treatment_rate  relative_lift_%    z_stat  \
visited_landing_page      0.636364        0.725874        14.065934  3.605125   
started_trial             0.251082        0.290909        15.862069  1.680318   
completed_onboarding      0.155844        0.212587        36.410256  2.743327   
converted_to_paid         0.031746        0.069930       120.279720  3.251871   

                       p_value  
visited_landing_page  0.000312  
started_trial         0.092895  
completed_onboarding  0.006082  
converted_to_paid     0.001146  


In [4]:
def prop_test(df, col, group_col='experiment_group'):
    c = df[df[group_col] == 'Control']
    t = df[df[group_col] == 'Treatment']
    x1, n1 = c[col].sum(), len(c)
    x2, n2 = t[col].sum(), len(t)
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p2 - p1) / se if se > 0 else np.nan
    pval = 2 * (1 - stats.norm.cdf(abs(z))) if se > 0 else np.nan
    lift = (p2 - p1) / p1 * 100 if p1 > 0 else np.inf
    return pd.Series({'control_rate': p1, 'treatment_rate': p2,
                       'relative_lift_%': lift, 'z_stat': z, 'p_value': pval})

funnel_cols = ['visited_landing_page', 'started_trial',
               'completed_onboarding', 'converted_to_paid']
funnel_results = pd.DataFrame({col: prop_test(df, col) for col in funnel_cols}).T
print(funnel_results)
# converted_to_paid: control ~3.2% -> treatment ~7.0%, p < 0.01 -> statistically significant

# ---- CELL 4: Guardrail metrics ----
c = df[df.experiment_group == 'Control']
t = df[df.experiment_group == 'Treatment']

# Support tickets (guardrail: should not increase)
tstat, p_tix = stats.ttest_ind(t['support_tickets_30d'], c['support_tickets_30d'], equal_var=False)
print(f"support_tickets_30d: control={c.support_tickets_30d.mean():.3f} "
      f"treatment={t.support_tickets_30d.mean():.3f} p={p_tix:.4f}")

# Refund rate (guardrail: should not increase)
refund_results = prop_test(df, 'refund_requested')
print("\nrefund_requested:\n", refund_results)

# Revenue per user overall (includes non-converters as 0)
tstat, p_rev = stats.ttest_ind(t['revenue_30d'], c['revenue_30d'], equal_var=False)
print(f"\nrevenue_30d (all users): control={c.revenue_30d.mean():.2f} "
      f"treatment={t.revenue_30d.mean():.2f} p={p_rev:.4f}")

# Revenue quality: average revenue PER CONVERTED USER
# (this is the key guardrail check - are the new conversions worth as much?)
conv = df[df.converted_to_paid == 1]
cc, tt = conv[conv.experiment_group == 'Control'], conv[conv.experiment_group == 'Treatment']
tstat, p_revconv = stats.ttest_ind(tt['revenue_30d'], cc['revenue_30d'], equal_var=False)
print(f"\nRevenue per converted user: control={cc.revenue_30d.mean():.2f} (n={len(cc)}), "
      f"treatment={tt.revenue_30d.mean():.2f} (n={len(tt)}), p={p_revconv:.4f}")

                      control_rate  treatment_rate  relative_lift_%    z_stat  \
visited_landing_page      0.636364        0.725874        14.065934  3.605125   
started_trial             0.251082        0.290909        15.862069  1.680318   
completed_onboarding      0.155844        0.212587        36.410256  2.743327   
converted_to_paid         0.031746        0.069930       120.279720  3.251871   

                       p_value  
visited_landing_page  0.000312  
started_trial         0.092895  
completed_onboarding  0.006082  
converted_to_paid     0.001146  
support_tickets_30d: control=0.219 treatment=0.372 p=0.0000

refund_requested:
 control_rate       0.000000
treatment_rate     0.004196
relative_lift_%         inf
z_stat             1.707015
p_value            0.087819
dtype: float64

revenue_30d (all users): control=51.75 treatment=53.88 p=0.9163

Revenue per converted user: control=1630.10 (n=22), treatment=770.41 (n=50), p=0.0791


In [5]:
# Needed to answer: should we launch for everyone, or only specific segments?
for seg in ['region', 'device_type', 'traffic_source', 'plan_type']:
    print(f"\n--- Conversion rate by {seg} ---")
    pivot = df.groupby([seg, 'experiment_group'])['converted_to_paid'] \
              .mean().unstack().round(4)
    pivot['lift_%'] = ((pivot['Treatment'] - pivot['Control']) / pivot['Control'] * 100).round(1)
    print(pivot)


--- Conversion rate by region ---
experiment_group  Control  Treatment  lift_%
region                                      
East               0.0253     0.0640   153.0
North              0.0345     0.0889   157.7
South              0.0326     0.0761   133.4
West               0.0338     0.0503    48.8

--- Conversion rate by device_type ---
experiment_group  Control  Treatment  lift_%
device_type                                 
Desktop            0.0450     0.0654    45.3
Mobile             0.0257     0.0734   185.6
Tablet             0.0179     0.0714   298.9

--- Conversion rate by traffic_source ---
experiment_group  Control  Treatment  lift_%
traffic_source                              
Email              0.0270     0.0714   164.4
Organic            0.0203     0.0622   206.4
Paid Search        0.0128     0.0625   388.3
Referral           0.0247     0.1099   344.9
Social             0.0769     0.0602   -21.7

--- Conversion rate by plan_type ---
experiment_group  Control  Treatme

In [6]:
print("""
DECISION FRAMEWORK
-------------------
1. Primary metric (converted_to_paid): Treatment wins, statistically significant.
2. Guardrails:
   - support_tickets_30d: significantly WORSE in Treatment.
   - revenue per converted user: notably LOWER in Treatment (directional, not
     fully significant at n=22 vs 50 - worth watching as more data comes in).
   - refund_requested: trending worse in Treatment.
3. Segment cuts: lift is NOT uniform - Social traffic source underperforms
   in Treatment, Basic plan shows almost no lift.

-> See README.md for the full recommendation.
""")


DECISION FRAMEWORK
-------------------
1. Primary metric (converted_to_paid): Treatment wins, statistically significant.
2. Guardrails:
   - support_tickets_30d: significantly WORSE in Treatment.
   - revenue per converted user: notably LOWER in Treatment (directional, not
     fully significant at n=22 vs 50 - worth watching as more data comes in).
   - refund_requested: trending worse in Treatment.
3. Segment cuts: lift is NOT uniform - Social traffic source underperforms
   in Treatment, Basic plan shows almost no lift.
 
-> See README.md for the full recommendation.

